# Практика · Семантична сегментація: FCN і DeepLab

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі сцени зошит малює формулами. Досить `torch`,
> `torchvision`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **30 мереж** (10 настройок × три зерна). Заміряно наскрізь:
> **665 секунд, тобто близько одинадцяти хвилин** — на чотирьох ядрах без
> відеокарти, в один потік, і при цьому машина була зайнята іншими розрахунками.
> Найдовший шматок — атрозна мережа: вона працює на нестиснутій карті 64×64,
> і це саме той час, який ми хочемо побачити своїми очима.

Тема про одну задачу: як зробити так, щоб мережа видала відповідь **того самого
розміру, що й вхід**. Що зробимо:

1. зберемо сцени 64×64 **з масками** й порахуємо, наскільки в них переважає фон;
2. напишемо **свої** `IoU`, `mIoU` і `Dice`, звіримо їх із прикладом, порахованим
   руками, і покажемо, чому точність по пікселях — не метрика;
3. поміряємо **без жодного навчання**, скільки маски вбиває сама сітка карти
   при кроці 2, 4, 8, 16 і 32;
4. напишемо **свою атрозну згортку** через `unfold` і звіримо її з
   `nn.Conv2d(dilation=)` через `np.allclose`;
5. навчимо FCN-подібні мережі з кроком 4, 8 і 16 — скільки коштує грубість;
6. порівняємо **три способи повернути роздільність**: білінійне збільшення,
   `ConvTranspose2d` і атрозні згортки без стискання;
7. покажемо **числом**, звідки береться артефакт шахівниці;
8. зберемо **ASPP** і порівняємо його з однією гілкою;
9. переберемо **три втрати** на дисбалансі 90 % фону;
10. порахуємо параметри справжніх `fcn_resnet50`, `deeplabv3_resnet50` і
    `lraspp_mobilenet_v3_large`.

In [ ]:
import math
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

print("torch   ", torch.__version__)
print("numpy   ", np.__version__)
print("потоків ", torch.get_num_threads())

## 1 · Сцени 64×64 — тепер із масками

Той самий наскрізний приклад, що й у блоці про детекцію: полотно 64 на 64 пікселі,
від одного до трьох предметів трьох класів, радіус 6-10 пікселів, шум зі стандартним
відхиленням 0.12.

Різниця одна: замість рамки ми зберігаємо **маску** — масив 64×64, у якому на місці
кожного пікселя стоїть номер класу. Нуль означає фон, одиниця — коло, двійка —
квадрат, трійка — трикутник. Саме з цих масок у блоці 5 виводились рамки, тож
розмітка тут істинна за побудовою, до пікселя.

In [ ]:
SIZE = 64                                   # сторона полотна в пікселях
CLASS_NAMES = ["фон", "коло", "квадрат", "трикутник"]
NUM_LABELS = len(CLASS_NAMES)               # три предмети + фон
OBJECT_CLASSES = range(1, NUM_LABELS)       # фон у mIoU не входить — див. розділ 2


def shape_mask(kind, center_x, center_y, radius):
    '''Маска однієї фігури на полотні 64×64 — та сама геометрія, що в блоці 5.'''
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                    # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                    # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def make_scene(rng, radius_low=6, radius_high=10):
    '''Одна сцена: зашумлена картинка і маска класів того самого розміру.'''
    image = np.zeros((SIZE, SIZE), np.float32)
    segmentation = np.zeros((SIZE, SIZE), np.int64)
    taken = []
    for _ in range(int(rng.integers(1, 4))):
        for _attempt in range(40):
            radius = int(rng.integers(radius_low, radius_high + 1))
            center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
            center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
            kind = int(rng.integers(0, 3))
            box = [center_x - radius, center_y - radius,
                   center_x + radius + 1, center_y + radius + 1]

            # не даємо предметам злипатись більше ніж на 45 % площі нового —
            # інакше маска перетворюється на одну суцільну пляму
            overlap_too_big = False
            for previous in taken:
                width = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                height = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if width * height > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    overlap_too_big = True
                    break
            if overlap_too_big:
                continue

            mask = shape_mask(kind, center_x, center_y, radius)
            image[mask] = 1.0
            segmentation[mask] = kind + 1          # 0 лишається фоном
            taken.append(box)
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return image, segmentation


def make_dataset(seed, count, radius_low=6, radius_high=10):
    '''Готовий набір одразу тензорами — щоб не перетворювати його в кожній епосі.'''
    rng = np.random.default_rng(seed)
    scenes = [make_scene(rng, radius_low, radius_high) for _ in range(count)]
    images = torch.from_numpy(np.stack([s[0] for s in scenes])[:, None])
    masks = torch.from_numpy(np.stack([s[1] for s in scenes]))
    return images, masks


started = time.time()
train_images, train_masks = make_dataset(42, 160)
test_images, test_masks = make_dataset(7, 120)
print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, форма картинок %s, форма масок %s"
      % (len(train_images), tuple(train_images.shape), tuple(train_masks.shape)))
print("перевірних сцен %d" % len(test_images))

### Як це виглядає

Зліва картинка, справа її маска. Зверни увагу: маска не має шуму й не має півтонів —
це просто номер класу в кожній клітинці.

In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(13, 5.4))
for column in range(5):
    axes[0, column].imshow(train_images[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title("сцена %d" % column, fontsize=9)
    axes[1, column].imshow(train_masks[column], cmap="viridis", vmin=0, vmax=3)
    axes[1, column].set_title("маска", fontsize=9)
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()
plt.show()
print("значення в масці:", sorted(set(train_masks[:5].reshape(-1).tolist())))

## 2 · Метрики: пишемо свої й звіряємо руками

Спершу порахуємо, з чого складається вибірка. Це не формальність: саме частка фону
пояснює, чому точність по пікселях нічого не варта.

In [ ]:
def class_shares(masks):
    '''Скільки пікселів кожного класу — у штуках і в частках.'''
    counts = torch.bincount(masks.reshape(-1), minlength=NUM_LABELS).double()
    return counts, counts / counts.sum()


for name, masks in (("навчальна", train_masks), ("перевірна", test_masks)):
    counts, shares = class_shares(masks)
    print("%s вибірка:" % name)
    for index in range(NUM_LABELS):
        print("   %-11s %9d  %6.2f %%"
              % (CLASS_NAMES[index], int(counts[index]), 100 * shares[index]))

### Матриця плутанини, IoU і mIoU — своя реалізація

Усі три метрики зручно рахувати з однієї таблиці: **матриці плутанини**. Це квадрат
розміру «класів × класів», у якому в клітинці (рядок `a`, стовпець `b`) стоїть
кількість пікселів, які насправді належать класу `a`, а мережа назвала їх `b`.

З неї виходить усе:

- перетин класу `c` — це діагональний елемент `cm[c, c]`;
- скільки пікселів класу `c` є в істині — сума рядка `c`;
- скільки пікселів мережа назвала класом `c` — сума стовпця `c`;
- обʼєднання — сума рядка плюс сума стовпця мінус діагональ (інакше перетин
  порахували б двічі).

In [ ]:
def confusion_matrix(prediction, truth):
    '''Матриця плутанини: рядок — істинний клас, стовпець — прогноз.'''
    # два індекси зводимо в один, щоб порахувати все одним bincount
    flat = truth.reshape(-1) * NUM_LABELS + prediction.reshape(-1)
    return torch.bincount(flat, minlength=NUM_LABELS ** 2).reshape(NUM_LABELS, NUM_LABELS)


def iou_per_class(confusion):
    '''IoU кожного класу окремо і розмір його обʼєднання (щоб знати, чи клас узагалі є).'''
    confusion = confusion.double()
    intersection = confusion.diag()
    union = confusion.sum(dim=1) + confusion.sum(dim=0) - intersection
    return (intersection / union.clamp(min=1)).numpy(), union.numpy()


def mean_iou(confusion):
    '''mIoU по трьох класах предметів. Фон свідомо не входить — див. пояснення нижче.'''
    values, union = iou_per_class(confusion)
    present = [values[c] for c in OBJECT_CLASSES if union[c] > 0]
    return float(np.mean(present)) if present else 0.0


def pixel_accuracy(confusion):
    '''Частка правильно підписаних пікселів.'''
    confusion = confusion.double()
    return float(confusion.diag().sum() / confusion.sum())


def dice_per_class(confusion):
    '''Коефіцієнт Дайса: подвоєний перетин, поділений на суму площ.'''
    confusion = confusion.double()
    intersection = confusion.diag()
    total = confusion.sum(dim=1) + confusion.sum(dim=0)
    return (2 * intersection / total.clamp(min=1)).numpy()


print("функції готові:", confusion_matrix, mean_iou)

### Звірка на прикладі, порахованому руками

Найнадійніша перевірка — приклад, у якому відповідь відома до запуску.

Істина: квадрат 10×10, тобто **100** пікселів класу «квадрат».
Прогноз: квадрат 12×12 = **144** пікселі, зсунутий на 2 пікселі вправо й на 2 вниз.

- перетин: зсув на 2 зʼїдає два стовпці й два рядки, лишається 8×8 = **64**;
- обʼєднання: 100 + 144 − 64 = **180**;
- IoU = 64 / 180 = **0.3556**;
- Dice = 2·64 / (100 + 144) = 128 / 244 = **0.5246**.

Перевіримо, що наші функції дають те саме, і заразом — що тотожність
`Dice = 2·IoU / (1 + IoU)` справді виконується.

In [ ]:
hand_truth = torch.zeros(30, 30, dtype=torch.long)
hand_truth[5:15, 5:15] = 2                      # квадрат 10×10 класу «квадрат»

hand_prediction = torch.zeros(30, 30, dtype=torch.long)
hand_prediction[7:19, 7:19] = 2                 # квадрат 12×12, зсунутий на 2 і 2

hand_confusion = confusion_matrix(hand_prediction, hand_truth)
values, union = iou_per_class(hand_confusion)
dice_values = dice_per_class(hand_confusion)

intersection = int(hand_confusion[2, 2])
print("перетин     %d  (рахували руками: 64)" % intersection)
print("обʼєднання  %d  (рахували руками: 180)" % int(union[2]))
print("IoU         %.4f  (рахували руками: 0.3556)" % values[2])
print("Dice        %.4f  (рахували руками: 0.5246)" % dice_values[2])

assert intersection == 64 and int(union[2]) == 180, "геометрія прикладу зламалась!"
assert np.isclose(values[2], 64 / 180), "наше IoU розійшлося з ручним підрахунком!"
assert np.isclose(dice_values[2], 2 * 64 / 244), "наш Dice розійшовся з ручним підрахунком!"

# і сама тотожність, яку ми цитуємо в лекції
from_formula = 2 * values[2] / (1 + values[2])
assert np.isclose(dice_values[2], from_formula), "тотожність Dice = 2·IoU/(1+IoU) не справдилась!"
print("✅ збігається, і формула Dice = 2·IoU/(1+IoU) дає %.4f" % from_formula)

### Чому точність по пікселях безглузда

Тепер найдешевша модель у світі: вона фарбує все зображення фоном. Порахуємо їй
обидві метрики.

In [ ]:
all_background = torch.zeros_like(test_masks)
background_confusion = confusion_matrix(all_background, test_masks)
values, _ = iou_per_class(background_confusion)
_, test_shares = class_shares(test_masks)

print("модель «усе фон» на перевірній вибірці:")
print("   точність по пікселях  %.4f" % pixel_accuracy(background_confusion))
print("   частка фону в істині  %.4f   ← те саме число, і це не збіг"
      % float(test_shares[0]))
print("   IoU по класах        ", np.round(values, 4))
print("   mIoU (без фону)       %.4f" % mean_iou(background_confusion))
print("   mIoU (якби з фоном)   %.4f" % float(np.mean(values)))
print()
print("Саме тому в курсі mIoU рахується без фону: з фоном порожня модель")
print("дістає майже чверть бала ні за що.")

### Наскільки Dice оптимістичніший за IoU

Візьмімо дві маски однакової площі й будемо міняти частку перекриття. Якщо
перекриття дорівнює `t` частини площі кожної маски, то IoU = t / (2 − t).

In [ ]:
print("перекриття |    IoU |   Dice | у скільки разів більший")
for overlap in (0.9, 0.5, 0.1):
    iou = overlap / (2 - overlap)
    dice = 2 * iou / (1 + iou)
    print("   %5.0f %% | %.4f | %.4f | %.2f" % (100 * overlap, iou, dice, dice / iou))
print()
print("На добрих масках різниця 10 %, на поганих — майже вдвічі.")
print("Читаючи чуже число, дивись, яку саме метрику назвали.")

## 3 · Скільки маски вбиває сама сітка — без жодного навчання

Перш ніж навчати мережі, поміряємо межу знизу. Візьмемо **істинну** маску, пропустимо
її крізь карту кроку `S` і повернемо назад до 64×64. Два способи повернення:

- **сходинками** — беремо мітку в лівому верхньому куті кожної клітинки й розтягуємо
  її на весь квадрат `S×S`;
- **плавно** — усереднюємо по клітинці ознаку кожного класу, розтягуємо ці чотири
  карти білінійно й беремо клас з найбільшим значенням.

Друге — рівно те, що робить справжній сегментатор в останньому рядку.

In [ ]:
def coarse_and_back_nearest(masks, stride):
    '''Загрубити сходинками й повернути: беремо кожен S-й піксель і розтягуємо.'''
    small = masks[:, ::stride, ::stride]
    return small.repeat_interleave(stride, dim=1).repeat_interleave(stride, dim=2)


def coarse_and_back_bilinear(masks, stride):
    '''Загрубити плавно: середнє по клітинці для кожного класу, потім білінійне збільшення.'''
    one_hot = F.one_hot(masks, NUM_LABELS).permute(0, 3, 1, 2).float()
    small = F.avg_pool2d(one_hot, stride)
    big = F.interpolate(small, size=(SIZE, SIZE), mode="bilinear", align_corners=False)
    return big.argmax(dim=1)


print("крок | карта  | mIoU сходинками | mIoU плавно | точність по пікселях")
ceiling = {}
for stride in (2, 4, 8, 16, 32):
    nearest = confusion_matrix(coarse_and_back_nearest(test_masks, stride), test_masks)
    smooth = confusion_matrix(coarse_and_back_bilinear(test_masks, stride), test_masks)
    ceiling[stride] = (mean_iou(nearest), mean_iou(smooth))
    print("%4d | %2dx%-2d  |          %.4f |      %.4f | %.4f"
          % (stride, SIZE // stride, SIZE // stride,
             ceiling[stride][0], ceiling[stride][1], pixel_accuracy(nearest)))
print()
print("Крок 8 лишає від маски дві третини при плавному поверненні й лише 0.40 при")
print("сходинках — тобто спосіб повернення важить майже стільки ж, скільки сам крок.")

## 4 · Своя атрозна згортка

Атрозна згортка (atrous, вона ж dilated) читає не сусідні пікселі, а кожен `d`-й.
Ваг у ній стільки ж, скільки у звичайній: девʼять на пару каналів при ядрі 3×3.

Напишемо її самі через `F.unfold`. Ця функція вирізає з картинки всі віконця, які
побачить ядро, і викладає їх стовпчиками — після цього згортка стає звичайним
множенням матриць. Аргумент `dilation` у `unfold` задає крок між відліками віконця,
тобто рівно те, що нам треба.

In [ ]:
def atrous_conv_by_hand(x, weight, bias, dilation):
    '''Атрозна згортка вручну: unfold + множення матриць.'''
    out_channels, in_channels, k, _ = weight.shape
    padding = dilation * (k - 1) // 2            # щоб розмір виходу не змінився
    # patches: (батч, in_channels*k*k, скільки позицій) — усі віконця стовпчиками
    patches = F.unfold(x, kernel_size=k, dilation=dilation, padding=padding)
    flat_weight = weight.reshape(out_channels, -1)
    result = flat_weight @ patches + bias[:, None]
    return result.reshape(x.shape[0], out_channels, x.shape[2], x.shape[3])


torch.manual_seed(0)
probe = torch.randn(2, 3, 16, 16)
print("dilation | збіг із nn.Conv2d | найбільша різниця")
for dilation in (1, 2, 4, 8):
    reference_layer = nn.Conv2d(3, 5, 3, padding=dilation, dilation=dilation)
    reference = reference_layer(probe)
    ours = atrous_conv_by_hand(probe, reference_layer.weight,
                               reference_layer.bias, dilation)
    same = np.allclose(ours.detach().numpy(), reference.detach().numpy(), atol=1e-5)
    assert same, "наша атрозна згортка розійшлася з бібліотечною при dilation %d!" % dilation
    print("%8d | %17s | %.2e"
          % (dilation, "так", float((ours - reference).abs().max())))
print("✅ збігається при всіх dilation — усередині бібліотеки магії немає")

### Головна таблиця теми: поле росте, параметри стоять

Тепер порахуємо, що ця операція дає задарма. Беремо один і той самий шар
`Conv2d(64, 64, 3, padding=d, dilation=d)` і міняємо лише `dilation`.

In [ ]:
print("dilation | ефективне ядро | поле, пікселів | параметрів | множень на піксель")
for dilation in (1, 2, 4, 8, 16):
    layer = nn.Conv2d(64, 64, 3, padding=dilation, dilation=dilation)
    parameters = sum(p.numel() for p in layer.parameters())
    effective = 2 * dilation + 1
    print("%8d | %10d×%-3d | %14d | %10d | %d"
          % (dilation, effective, effective, effective * effective,
             parameters, 64 * 64 * 9))
print()
print("Поле зору виросло з 9 пікселів до 1089 — у 121 раз. Параметрів як було")
print("36 928, так і лишилось. Це вся ідея DeepLab в одній таблиці.")

## 5 · Мережі

Одне тіло на всі досліди — три-пʼять згорткових блоків по 16 каналів. Міняємо в ньому
рівно одну річ за раз:

- `Downsampling` стискає карту в `stride` разів пулінгом, а роздільність повертає
  або білінійним збільшенням, або стосом `ConvTranspose2d`;
- `Atrous` не стискає взагалі: те саме число шарів, але поле зору росте через
  `dilation`;
- `ASPPHead` — кілька атрозних гілок паралельно плюс гілка глобального усереднення.

Ініціалізацію задаємо руками для всіх однаково. Це важливо: замовчування PyTorch для
згорток занижене, і на таких маленьких мережах різниця в ініціалізації переважила б
різницю між архітектурами.

In [ ]:
def conv_block(in_channels, out_channels, dilation=1):
    '''Згортка 3×3 з батчнормом і ReLU. dilation=1 дає звичайну згортку.'''
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=dilation,
                  dilation=dilation, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))


class Downsampling(nn.Module):
    '''Тіло зі стисканням у stride разів; спосіб повернення розміру задається аргументом.'''

    def __init__(self, stride=8, channels=16, upsample="bilinear", deconv_kernel=4):
        super().__init__()
        levels = int(math.log2(stride))
        layers = [conv_block(1, channels)]
        for _ in range(levels):
            layers += [nn.MaxPool2d(2), conv_block(channels, channels)]
        self.body = nn.Sequential(*layers)
        self.upsample = upsample
        self.head = nn.Conv2d(channels, NUM_LABELS, 1)     # голова 1×1 замість FC
        self.deconv = None
        if upsample == "deconv":
            k = deconv_kernel
            # padding і output_padding підібрані так, щоб вихід був рівно вдвічі
            # більшим за вхід при будь-якому ядрі — інакше різниця між ядрами
            # мішалася б із різницею розмірів
            padding = (k - 2) // 2 if k % 2 == 0 else (k - 1) // 2
            output_padding = 0 if k % 2 == 0 else 1
            stages = []
            for _ in range(levels):
                stages += [nn.ConvTranspose2d(channels, channels, k, stride=2,
                                              padding=padding,
                                              output_padding=output_padding, bias=False),
                           nn.BatchNorm2d(channels), nn.ReLU(inplace=True)]
            self.deconv = nn.Sequential(*stages)

    def forward(self, x):
        size = x.shape[-2:]
        features = self.body(x)
        if self.upsample == "bilinear":
            logits = self.head(features)
            return F.interpolate(logits, size=size, mode="bilinear", align_corners=False)
        return self.head(self.deconv(features))


class Atrous(nn.Module):
    '''Те саме число шарів, але жодного пулінгу: поле зору росте через dilation.'''

    def __init__(self, dilations=(1, 2, 4, 8, 16), channels=16):
        super().__init__()
        layers = [conv_block(1, channels, dilations[0])]
        for dilation in dilations[1:]:
            layers.append(conv_block(channels, channels, dilation))
        self.body = nn.Sequential(*layers)
        self.head = nn.Conv2d(channels, NUM_LABELS, 1)

    def forward(self, x):
        return self.head(self.body(x))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print("крок  4:", count_parameters(Downsampling(4)), "параметрів")
print("крок  8:", count_parameters(Downsampling(8)), "параметрів")
print("крок 16:", count_parameters(Downsampling(16)), "параметрів")
print("атрозне тіло з пʼяти шарів:", count_parameters(Atrous()), "параметрів",
      "← стільки ж, скільки в тілі кроку 16")

In [ ]:
class ASPPHead(nn.Module):
    '''Кілька атрозних гілок паралельно плюс гілка глобального усереднення.'''

    def __init__(self, channels=16, branch=16, dilations=(1, 2, 4), pooling=True):
        super().__init__()
        self.branches = nn.ModuleList()
        for dilation in dilations:
            if dilation == 1:
                # гілка без контексту: чисте перекодування ознак у точці
                self.branches.append(nn.Sequential(
                    nn.Conv2d(channels, branch, 1, bias=False),
                    nn.BatchNorm2d(branch), nn.ReLU(inplace=True)))
            else:
                self.branches.append(conv_block(channels, branch, dilation))
        self.pooling = None
        if pooling:
            # уся карта одним числом на канал — контекст, якого не бачить жодне dilation
            self.pooling = nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, branch, 1, bias=False),
                nn.ReLU(inplace=True))
        parts = len(dilations) + (1 if pooling else 0)
        self.project = nn.Sequential(
            nn.Conv2d(branch * parts, channels, 1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        pieces = [branch(x) for branch in self.branches]
        if self.pooling is not None:
            pooled = self.pooling(x)
            pieces.append(pooled.expand(-1, -1, x.shape[-2], x.shape[-1]))
        return self.project(torch.cat(pieces, dim=1))


class WithContext(nn.Module):
    '''Тіло зі стисканням → модуль контексту → голова 1×1 → білінійне збільшення.'''

    def __init__(self, context, channels=16, stride=16):
        super().__init__()
        layers = [conv_block(1, channels)]
        for _ in range(int(math.log2(stride))):
            layers += [nn.MaxPool2d(2), conv_block(channels, channels)]
        self.body = nn.Sequential(*layers)
        self.context = context
        self.head = nn.Conv2d(channels, NUM_LABELS, 1)

    def forward(self, x):
        size = x.shape[-2:]
        logits = self.head(self.context(self.body(x)))
        return F.interpolate(logits, size=size, mode="bilinear", align_corners=False)


print("одна гілка dilation 2:",
      count_parameters(WithContext(ASPPHead(dilations=(2,), pooling=False))), "параметрів")
print("ASPP 1·2·4 + усереднення:",
      count_parameters(WithContext(ASPPHead(dilations=(1, 2, 4), pooling=True))), "параметрів")

### Навчання і оцінювання

Один рецепт на всі мережі: AdamW, швидкість навчання 3·10⁻³ з косинусним спаданням,
14 епох, батч 20. Кожну настройку проганяємо на **трьох зернах** — різниця, менша за
розкид від зерна, не є різницею.

In [ ]:
EPOCHS, BATCH, LEARNING_RATE, SEEDS = 14, 20, 3e-3, (0, 1, 2)


def initialise(model, seed):
    '''Однаковий старт для всіх мереж зошита — інакше порівнюємо не архітектури,
    а щедрість початкових ваг.'''
    torch.manual_seed(seed)
    for module in model.modules():
        if hasattr(module, "reset_parameters"):
            module.reset_parameters()
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)


def train_model(model, images, masks, seed=0, loss_function=None):
    '''Навчає мережу й повертає витрачений час у секундах.'''
    initialise(model, seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    steps = EPOCHS * math.ceil(len(images) / BATCH)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=steps)
    order = torch.arange(len(images))
    started = time.time()
    for _ in range(EPOCHS):
        order = order[torch.randperm(len(order))]      # свій порядок у кожній епосі
        for start in range(0, len(images), BATCH):
            chosen = order[start:start + BATCH]
            logits = model(images[chosen])
            if loss_function is None:
                loss = F.cross_entropy(logits, masks[chosen])
            else:
                loss = loss_function(logits, masks[chosen])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            schedule.step()
    return time.time() - started


def evaluate(model, images, masks, batch=40):
    '''mIoU і точність по пікселях на всій вибірці.'''
    model.eval()
    total = torch.zeros(NUM_LABELS, NUM_LABELS, dtype=torch.long)
    with torch.no_grad():
        for start in range(0, len(images), batch):
            prediction = model(images[start:start + batch]).argmax(dim=1)
            total += confusion_matrix(prediction, masks[start:start + batch])
    model.train()
    return mean_iou(total), pixel_accuracy(total), total


RESULTS = {}


def measure(tag, build_model, loss_function=None):
    '''Три зерна однієї настройки: середнє mIoU, розкид, час.'''
    scores, times = [], []
    for seed in SEEDS:
        model = build_model()
        times.append(train_model(model, train_images, train_masks, seed, loss_function))
        score, _accuracy, _matrix = evaluate(model, test_images, test_masks)
        scores.append(score)
    RESULTS[tag] = {"params": count_parameters(model),
                    "miou": float(np.mean(scores)), "spread": float(np.std(scores)),
                    "secs": float(np.mean(times)), "seeds": [round(s, 4) for s in scores]}
    print("%-26s параметрів %-7d %5.1f с  mIoU %.4f ± %.4f   по зернах %s"
          % (tag, count_parameters(model), np.mean(times), np.mean(scores),
             np.std(scores), " ".join("%.4f" % s for s in scores)), flush=True)
    return RESULTS[tag]


print("готово; далі почнуться навчання")

## 6 · Скільки коштує грубість: крок 4, 8 і 16

Перший замір. Тіло, голова `1×1`, білінійне збільшення до 64×64. Змінна одна — крок
тіла. Це і є FCN у мініатюрі.

In [ ]:
notebook_started = time.time()
for stride in (4, 8, 16):
    measure("білінійне, крок %d" % stride, lambda s=stride: Downsampling(s))
print()
print("Стеля з розділу 3 для порівняння (плавне повернення істинної маски):")
for stride in (4, 8, 16):
    print("   крок %2d: %.4f" % (stride, ceiling[stride][1]))

## 7 · Артефакт шахівниці — числом

Перш ніж навчати транспоновану згортку, подивимось на її геометрію. Покладімо всі
ваги ядра рівними одиниці й подамо на вхід одиниці. Тоді значення кожної вихідної
клітинки дорівнює просто **кількості відбитків ядра, які на неї впали**.

Якщо ця кількість різна для сусідніх клітинок — на виході буде регулярний візерунок,
і жодне навчання його не прибере: покриття задане геометрією, а не вагами.

In [ ]:
print("ядро | покриття в клітинці 2×2 | дисперсія | найбільше/найменше | ділиться на крок")
for kernel in (2, 3, 4, 5, 6):
    layer = nn.ConvTranspose2d(1, 1, kernel, stride=2, bias=False)
    with torch.no_grad():
        layer.weight.fill_(1.0)
    coverage = layer(torch.ones(1, 1, 8, 8))[0, 0]
    # відступаємо від країв: там покриття менше просто через межу картинки
    core = coverage[kernel:-kernel, kernel:-kernel]
    cells = [float(core[i::2, j::2].mean()) for i in (0, 1) for j in (0, 1)]
    print("%4d | %23s | %9.4f | %18.2f | %s"
          % (kernel, " ".join("%.0f" % c for c in cells), float(np.var(cells)),
             max(cells) / min(cells), "так" if kernel % 2 == 0 else "ні"))
print()
print("Правило видно одразу: ядро, кратне кроку, покриває вихід рівно.")
print("Ядро 3 при кроці 2 дає 1 · 2 · 2 · 4 — різниця вчетверо між сусідами.")

## 8 · Три способи повернути роздільність

Тепер порівняння, заради якого все затівалось. Беремо **найгірший** випадок із
розділу 6 — крок 16, тобто карту 4×4 — і лагодимо його трьома способами:

1. **білінійне збільшення** — нуль ваг;
2. **`ConvTranspose2d`** — чотири навчені сходинки по два;
3. **атрозне тіло** — стільки ж шарів, стільки ж параметрів, стільки ж поле зору,
   але **жодного пулінгу**: карта весь час лишається 64×64.

Третій варіант підібраний так, щоб порівняння було чесним: `Atrous((1,2,4,8,16))` має
рівно ті самі 9 588 параметрів і те саме поле зору 63 пікселі, що й тіло кроку 16.
Різниця тільки в тому, стискаємо ми чи ні.

In [ ]:
for kernel in (4, 3):
    measure("ConvTranspose, ядро %d" % kernel,
            lambda k=kernel: Downsampling(16, upsample="deconv", deconv_kernel=k))
measure("атрозне, без стискання", lambda: Atrous(dilations=(1, 2, 4, 8, 16)))
print()
print("Для порівняння — білінійне збільшення з того самого тіла кроку 16:")
print("   mIoU %.4f ± %.4f, %.1f с"
      % (RESULTS["білінійне, крок 16"]["miou"], RESULTS["білінійне, крок 16"]["spread"],
         RESULTS["білінійне, крок 16"]["secs"]))

## 9 · ASPP проти однієї гілки

Модуль контексту ставимо на те саме тіло кроку 16. Порівнюємо одну атрозну гілку
з `dilation` 2 і повний ASPP: три гілки (`dilation` 1, 2 і 4) плюс гілка глобального
усереднення.

In [ ]:
measure("одна гілка, dilation 2",
        lambda: WithContext(ASPPHead(dilations=(2,), pooling=False), stride=16))
measure("ASPP 1·2·4 + усереднення",
        lambda: WithContext(ASPPHead(dilations=(1, 2, 4), pooling=True), stride=16))

## 10 · Втрата на дисбалансі

Девʼяносто відсотків пікселів — фон, тож у звичайній крос-ентропії девʼять із десяти
доданків стосуються фону. Два звичні ліки: зважити класи обернено до їхньої частоти
або взяти втрату Дайса, яка нормована на площу класу й тому не залежить від його
розміру.

Міряємо на тілі кроку 8 — не найкращому й не найгіршому.

In [ ]:
train_counts, _ = class_shares(train_masks)
class_weight = (train_counts.sum() / (NUM_LABELS * train_counts)).float()
print("ваги класів:", {CLASS_NAMES[i]: round(float(class_weight[i]), 3)
                       for i in range(NUM_LABELS)})


def weighted_cross_entropy(logits, target):
    '''Крос-ентропія, у якій рідкісний клас коштує дорожче.'''
    return F.cross_entropy(logits, target, weight=class_weight)


def dice_loss(logits, target, smooth=1.0):
    '''Гладка версія Dice: замість жорстких міток — імовірності softmax.'''
    probabilities = F.softmax(logits, dim=1)
    truth = F.one_hot(target, NUM_LABELS).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)                                  # усе, крім осі класів
    intersection = (probabilities * truth).sum(dims)
    total = probabilities.sum(dims) + truth.sum(dims)
    return 1.0 - ((2 * intersection + smooth) / (total + smooth)).mean()


measure("зважена крос-ентропія", lambda: Downsampling(8), weighted_cross_entropy)
measure("втрата Дайса", lambda: Downsampling(8), dice_loss)
print()
print("Для порівняння — звичайна крос-ентропія на тому самому тілі:")
print("   mIoU %.4f ± %.4f"
      % (RESULTS["білінійне, крок 8"]["miou"], RESULTS["білінійне, крок 8"]["spread"]))

## 11 · Підсумкова таблиця зошита

Усе, що ми наміряли, в одному місці. Саме ці числа стоять у лекції.

In [ ]:
print("%-26s %9s %8s %9s %8s" % ("настройка", "парам.", "час, с", "mIoU", "розкид"))
for tag, row in RESULTS.items():
    print("%-26s %9d %8.1f %9.4f %8.4f"
          % (tag, row["params"], row["secs"], row["miou"], row["spread"]))
print()
print("усього на навчання пішло %.0f с" % (time.time() - notebook_started))
print()
print(json.dumps(RESULTS, ensure_ascii=False))

## 12 · Справжні моделі з torchvision

Наостанок — скільки важать готові сегментатори. Будуємо їх **без ваг**
(`weights=None, weights_backbone=None`): нам потрібна сама конструкція, а не
завантаження з мережі.

In [ ]:
from torchvision.models import segmentation as tv_segmentation

print("%-32s %12s %12s %12s" % ("модель", "тіло", "голова", "усього"))
for name in ("fcn_resnet50", "deeplabv3_resnet50",
             "lraspp_mobilenet_v3_large", "deeplabv3_mobilenet_v3_large"):
    model = getattr(tv_segmentation, name)(weights=None, weights_backbone=None)
    parts = {child_name: count_parameters(child)
             for child_name, child in model.named_children()}
    print("%-32s %12d %12d %12d"
          % (name, parts["backbone"], parts["classifier"], count_parameters(model)))
print()
print("Перші дві моделі мають ОДНАКОВЕ тіло — уся різниця між FCN і DeepLab")
print("сидить у голові: ASPP коштує 6.7 мільйона ваг.")
print("LR-ASPP легша за DeepLab на ResNet-50 у %.1f раза."
      % (count_parameters(tv_segmentation.deeplabv3_resnet50(weights=None, weights_backbone=None))
         / count_parameters(tv_segmentation.lraspp_mobilenet_v3_large(weights=None,
                                                                     weights_backbone=None))))

## Завдання

**🟢 Рівень 1.** Додай у розділ 3 крок 3 і крок 6 (карта 21×21 і 10×10 після
обрізання). Чи лягають вони на ту саму криву, що степені двійки? Побудуй графік
`mIoU` від кроку для обох способів повернення.

**🟡 Рівень 2.** Візьми мережу кроку 8 і поміряй `mIoU` **окремо на смузі межі**
завширшки 2 пікселі. Смугу шукай так:

```python
def boundary_band(masks, width=2):
    y = masks.unsqueeze(1).float()
    biggest = F.max_pool2d(y, 2 * width + 1, 1, width)
    smallest = -F.max_pool2d(-y, 2 * width + 1, 1, width)
    return (biggest != smallest).squeeze(1)      # де максимум і мінімум околу різні
```

Наскільки метрика на межі нижча за метрику на всьому зображенні?

**🔴 Рівень 3.** Зроби чесне порівняння ASPP з однією гілкою **за однакового
бюджету параметрів**: розшир одну гілку (аргумент `branch`), поки її мережа не
матиме стільки ж ваг, скільки мережа з ASPP. Три зерна. Чи лишається виграш ASPP,
коли справа не в кількості ваг?